In [2]:
!pip install -q llama-cpp-python huggingface-hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 36.8 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00


In [3]:
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
import os

In [5]:
# Download the GGUF model file
# The repository has multiple quantized versions. Choose based on your needs:
# - Smaller models (Q2_K, Q3_K, Q4_K): Faster, less memory, lower quality
# - Larger models (Q5_K, Q6_K, Q8_0): Slower, more memory, higher quality
# - f16: Full precision, largest size, best quality

# Recommended: Q4_K_M for good balance of quality and speed
model_filename = "finance-chat-Q4_K_M.gguf"  

# Other options (uncomment to use):
# model_filename = "finance-chat-Q3_K_M.gguf"  # Smaller, faster
# model_filename = "finance-chat-Q5_K_M.gguf"  # Larger, better quality
# model_filename = "finance-chat-Q8_0.gguf"     # High quality

model_path = hf_hub_download(
    repo_id="andrijdavid/finance-chat-GGUF",
    filename=model_filename,
    repo_type="model"
)

print(f"Model downloaded: {model_filename}")
print(f"Path: {model_path}")

finance-chat-Q4_K_M.gguf:   0%|          | 0.00/4.08G [00:00<?, ?B/s]

Model downloaded: finance-chat-Q4_K_M.gguf
Path: /root/.cache/huggingface/hub/models--andrijdavid--finance-chat-GGUF/snapshots/d24722df135ecb014356469fc23dfc4ea4aab2c0/finance-chat-Q4_K_M.gguf


In [6]:
# Initialize the model with llama.cpp
# Adjust parameters based on your Kaggle environment
llm = Llama(
    model_path=model_path,
    n_ctx=4096,  # Context window (increased for better conversation)
    n_threads=2,  # Number of CPU threads (Kaggle usually has 2-4 cores)
    n_gpu_layers=0,  # Set to 35-40 if GPU is enabled in Kaggle
    n_batch=512,  # Batch size for prompt processing
    verbose=False
)

print("✓ Finance Chat Model loaded successfully!")
print(f"Model: {model_filename}")
print(f"Context window: 4096 tokens")

✓ Finance Chat Model loaded successfully!
Model: finance-chat-Q4_K_M.gguf
Context window: 4096 tokens


In [7]:
def chat_with_model(prompt, max_tokens=512, temperature=0.7, top_p=0.9):
    """
    Send a prompt to the model and get a response
    
    Args:
        prompt: The input text
        max_tokens: Maximum tokens to generate
        temperature: Randomness (0.1=focused, 1.0=creative)
        top_p: Nucleus sampling parameter
    """
    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        repeat_penalty=1.1,  # Reduce repetition
        stop=["User:", "Human:", "\n\n\n"],  # Stop tokens
        echo=False
    )
    
    return response['choices'][0]['text'].strip()

print("✓ Chat function ready!")

✓ Chat function ready!


In [8]:
# Conversation history with system prompt
system_prompt = "You are a helpful financial advisor assistant. Provide clear, accurate, and helpful information about finance, investing, and money management."
conversation_history = [f"System: {system_prompt}"]

print("=" * 60)
print("💬 Finance Chat Bot - Interactive Mode")
print("=" * 60)
print("Ask me anything about finance, investing, or money!")
print("Commands: 'quit'=exit, 'clear'=reset chat, 'history'=show chat")
print("=" * 60)

while True:
    user_input = input("\n🧑 You: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("👋 Goodbye! Happy investing!")
        break
    
    if user_input.lower() == 'clear':
        conversation_history = [f"System: {system_prompt}"]
        print("🔄 Chat history cleared!")
        continue
    
    if user_input.lower() == 'history':
        print("\n📜 Conversation History:")
        for msg in conversation_history[1:]:  # Skip system prompt
            print(f"  {msg}")
        continue
    
    # Build the prompt with conversation history
    prompt = "\n".join(conversation_history) + f"\nUser: {user_input}\nAssistant:"
    
    # Get model response
    try:
        response = chat_with_model(prompt, temperature=0.7, max_tokens=512)
        
        # Update conversation history
        conversation_history.append(f"User: {user_input}")
        conversation_history.append(f"Assistant: {response}")
        
        # Keep only last 8 exchanges (16 messages + system prompt)
        if len(conversation_history) > 17:
            conversation_history = [conversation_history[0]] + conversation_history[-16:]
        
        print(f"\n🤖 Assistant: {response}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Try a shorter message or type 'clear' to reset.")

💬 Finance Chat Bot - Interactive Mode
Ask me anything about finance, investing, or money!
Commands: 'quit'=exit, 'clear'=reset chat, 'history'=show chat



🧑 You:  hello



🤖 Assistant: Hello there! How can I assist you today?



🧑 You:  Create a 60-second pitch for a new D2C skincare brand, budget $30k/month, target women 22–35 in tier-1 Indian cities, key channel Instagram Reels, goal is 20% increase in monthly online sales.



🤖 Assistant: Thank you for reaching out! To create a 60-second pitch for the new D2C skincare brand, I will firstly analyze the target audience and their preferences. Then, I will suggest creating a unique selling proposition (USP) that highlights the benefits of using your products. Additionally, I will recommend developing an engaging Instagram Reels strategy to reach out to the target audience. Finally, I will provide you with a budget breakdown for $30k/month and suggest a plan to increase online sales by 20%.


...


🧑 You:  up to you



🤖 Assistant: Great, I will suggest creating short videos that showcase the benefits of using your products.



🧑 You:  ok now generate marketing speech



🤖 Assistant: Sure, here's a possible 60-second pitch:
"Hey there! Are you tired of using skincare products that don't work? Our new D2C brand is here to change that. We have created products that are not only effective but also affordable and easy to use. With our unique blend of natural ingredients, your skin will look healthier and younger than ever before. Plus, our Instagram Reels strategy will keep you entertained while learning about the benefits of using our products. So why wait? Try our skincare brand today and see the difference for yourself!"


KeyboardInterrupt: Interrupted by user